**1. Imports and Setup**

In [25]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import random
import time

**2. Dataset Class**

In [26]:
class EyeBoundingBoxDataset(Dataset):
    def __init__(self, subject_ids, root_dir, transform=None):
        self.root_dir = root_dir
        self.subject_ids = subject_ids
        self.transform = transform
        self.data = []

        for subject_id in self.subject_ids:
            subject_dir = os.path.join(root_dir, 'openEDS', 'openEDS', subject_id)
            bbox_file = os.path.join(root_dir, 'bbox', 'bbox', f"{subject_id}.txt")
            with open(bbox_file, 'r') as f:
                bboxes = [list(map(float, line.strip().split())) for line in f.readlines()]
            for i in range(len(bboxes) - 1):
                img0_path = os.path.join(subject_dir, f"{i}.png")
                img1_path = os.path.join(subject_dir, f"{i+1}.png")
                self.data.append((img0_path, img1_path, bboxes[i+1]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img0_path, img1_path, bbox = self.data[idx]
        img0 = Image.open(img0_path).convert('L')
        img1 = Image.open(img1_path).convert('L')

        if self.transform:
            img0 = self.transform(img0)
            img1 = self.transform(img1)

        diff = img1 - img0
        input_tensor = torch.cat((img1, diff), dim=0)
        target = torch.tensor(bbox, dtype=torch.float32)
        return input_tensor, target

In [27]:
# Define the root directory of your dataset
root_dir = '/Users/omaraguilarjr/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/openEDS2019'

# Example usage to load a dataset
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Example loading dataset with specific subjects
subject_ids = ['S_0', 'S_1']
dataset = EyeBoundingBoxDataset(subject_ids, root_dir, transform)

# Checking the length and a sample
print(f"Total samples in dataset: {len(dataset)}")
sample_input, sample_target = dataset[0]
print(f"Sample Input Shape: {sample_input.shape}")
print(f"Sample Target: {sample_target}")

Total samples in dataset: 305
Sample Input Shape: torch.Size([2, 64, 64])
Sample Target: tensor([162., 551., 167., 329.])


**3. Model Definition**

In [28]:
class LightweightBBoxCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 4)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x

In [29]:
# Initialize the model
model = LightweightBBoxCNN()
print(model)

LightweightBBoxCNN(
  (features): Sequential(
    (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (regressor): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=4, bias=True)
  )
)


**4. Subject-Based Train/Validation/Test Split**

In [30]:
def split_subjects_and_save(root_dir, output_file='subject_split.txt', seed=42):
    random.seed(seed)
    subject_path = os.path.join(root_dir, 'openEDS', 'openEDS')
    all_subjects = sorted([d for d in os.listdir(subject_path) if d.startswith('S_') and os.path.isdir(os.path.join(subject_path, d))])

    random.shuffle(all_subjects)
    n_total = len(all_subjects)
    n_train = int(0.7 * n_total)
    n_val = int(0.2 * n_total)

    train_subjects = all_subjects[:n_train]
    val_subjects = all_subjects[n_train:n_train + n_val]
    test_subjects = all_subjects[n_train + n_val:]

    with open(output_file, 'w') as f:
        f.write("Training Subjects:\n")
        for s in train_subjects:
            f.write(f"{s}\n")
        f.write("\nValidation Subjects:\n")
        for s in val_subjects:
            f.write(f"{s}\n")
        f.write("\nTest Subjects:\n")
        for s in test_subjects:
            f.write(f"{s}\n")

    print(f"Subject split saved to {output_file}")
    return train_subjects, val_subjects, test_subjects

In [31]:
train_subjects, val_subjects, test_subjects = split_subjects_and_save(
    root_dir=root_dir,
    output_file='subject_split.txt'
)

Subject split saved to subject_split.txt


**5. Training Pipeline**

In [32]:
def train_model_by_subject(root_dir, train_subjects, val_subjects, num_epochs=20, batch_size=16, lr=1e-3):
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])

    train_dataset = EyeBoundingBoxDataset(train_subjects, root_dir, transform)
    val_dataset = EyeBoundingBoxDataset(val_subjects, root_dir, transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LightweightBBoxCNN().to(device)
    criterion = nn.SmoothL1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)
        
        avg_train_loss = train_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
        
        avg_val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(avg_val_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")
    
    return model, train_losses, val_losses

In [33]:
# Train the model
model, train_losses, val_losses = train_model_by_subject(
    root_dir=root_dir,
    train_subjects=train_subjects,
    val_subjects=val_subjects,
    num_epochs=20,
    batch_size=16,
    lr=1e-3
)

Epoch 1/20 - Train Loss: 35.4779 - Val Loss: 25.2221


KeyboardInterrupt: 

**6. Visualization of Training Loss**

In [ ]:
def plot_training_curve(train_losses, val_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Plot training and validation loss curves
plot_training_curve(train_losses, val_losses)

**7. Single Frame Prediction Visualization**

In [ ]:
def visualize_prediction(subject_id, img_idx, model, root_dir):
    model.eval()
    subject_str = f"S_{subject_id}"
    img_dir = os.path.join(root_dir, 'openEDS', 'openEDS', subject_str)
    bbox_file = os.path.join(root_dir, 'bbox', 'bbox', f"{subject_str}.txt")

    with open(bbox_file, 'r') as f:
        bboxes = [list(map(float, line.strip().split())) for line in f.readlines()]

    img0 = Image.open(os.path.join(img_dir, f"{img_idx}.png")).convert('L')
    img1 = Image.open(os.path.join(img_dir, f"{img_idx+1}.png")).convert('L')
    img1_np = np.array(img1)

    transform = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
    img0_t = transform(img0)
    img1_t = transform(img1)
    diff = img1_t - img0_t
    input_tensor = torch.cat((img1_t, diff), dim=0).unsqueeze(0).to(next(model.parameters()).device)

    pred_box = model(input_tensor).squeeze(0).detach().cpu().numpy()
    gt_box = bboxes[img_idx + 1]

    plt.figure(figsize=(8, 6))
    plt.imshow(img1_np, cmap='gray')
    plt.plot([gt_box[0], gt_box[1], gt_box[1], gt_box[0], gt_box[0]],
             [gt_box[2], gt_box[2], gt_box[3], gt_box[3], gt_box[2]], 'g-', label='Ground Truth')
    plt.plot([pred_box[0], pred_box[1], pred_box[1], pred_box[0], pred_box[0]],
             [pred_box[2], pred_box[2], pred_box[3], pred_box[3], pred_box[2]], 'r--', label='Prediction')
    plt.title(f"Subject {subject_id} Frame {img_idx + 1}")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Visualize prediction for a single frame
visualize_prediction(
    subject_id=0, 
    img_idx=42, 
    model=model, 
    root_dir=root_dir
)

**8. Animation Visualization of Predictions**

In [ ]:
def display_prediction_animation(subject_id, model, root_dir, fps=10):
    import matplotlib.animation as animation
    from IPython.display import HTML

    model.eval()
    subject_str = f"S_{subject_id}"
    img_dir = os.path.join(root_dir, 'openEDS', 'openEDS', subject_str)
    bbox_file = os.path.join(root_dir, 'bbox', 'bbox', f"{subject_str}.txt")

    with open(bbox_file, 'r') as f:
        bboxes = [list(map(float, line.strip().split())) for line in f.readlines()]

    transform = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
    fig = plt.figure()
    ims = []

    for idx in range(len(bboxes) - 1):
        img0 = Image.open(os.path.join(img_dir, f"{idx}.png")).convert('L')
        img1 = Image.open(os.path.join(img_dir, f"{idx+1}.png")).convert('L')
        img1_np = np.array(img1)

        img0_t = transform(img0)
        img1_t = transform(img1)
        diff = img1_t - img0_t
        input_tensor = torch.cat((img1_t, diff), dim=0).unsqueeze(0).to(next(model.parameters()).device)

        pred_box = model(input_tensor).squeeze(0).detach().cpu().numpy()
        gt_box = bboxes[idx + 1]

        im = plt.imshow(img1_np, cmap='gray', animated=True)
        gt_line, = plt.plot([gt_box[0], gt_box[1], gt_box[1], gt_box[0], gt_box[0]],
                            [gt_box[2], gt_box[2], gt_box[3], gt_box[3], gt_box[2]], 'g-')
        pr_line, = plt.plot([pred_box[0], pred_box[1], pred_box[1], pred_box[0], pred_box[0]],
                            [pred_box[2], pred_box[2], pred_box[3], pred_box[3], pred_box[2]], 'r--')
        ims.append([im, gt_line, pr_line])

    ani = animation.ArtistAnimation(fig, ims, interval=1000 // fps, blit=True)
    plt.close(fig)
    return HTML(ani.to_jshtml())

In [ ]:
# Display prediction animation for a subject
display_prediction_animation(
    subject_id=0,
    model=model,
    root_dir=root_dir,
    fps=10
)

**9. Bar Graph Comparison of Average Bounding Boxes**

In [ ]:
def plot_avg_bbox_comparison(subject_id, model, root_dir):
    model.eval()
    subject_str = f"S_{subject_id}"
    img_dir = os.path.join(root_dir, 'openEDS', 'openEDS', subject_str)
    bbox_file = os.path.join(root_dir, 'bbox', 'bbox', f"{subject_str}.txt")

    with open(bbox_file, 'r') as f:
        bboxes = [list(map(float, line.strip().split())) for line in f.readlines()]

    transform = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
    preds = []

    for idx in range(len(bboxes) - 1):
        img0 = Image.open(os.path.join(img_dir, f"{idx}.png")).convert('L')
        img1 = Image.open(os.path.join(img_dir, f"{idx+1}.png")).convert('L')
        img0_t = transform(img0)
        img1_t = transform(img1)
        diff = img1_t - img0_t
        input_tensor = torch.cat((img1_t, diff), dim=0).unsqueeze(0).to(next(model.parameters()).device)
        with torch.no_grad():
            pred = model(input_tensor).squeeze(0).cpu().numpy()
        preds.append(pred)

    preds = np.array(preds)
    gts = np.array(bboxes[1:])
    avg_pred = preds.mean(axis=0)
    avg_gt = gts.mean(axis=0)

    labels = ['xmin', 'xmax', 'ymin', 'ymax']
    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots()
    ax.bar(x - width / 2, avg_gt, width, label='Ground Truth')
    ax.bar(x + width / 2, avg_pred, width, label='Predicted')
    ax.set_ylabel('Average Value')
    ax.set_title(f'Average Bounding Box for Subject {subject_id}')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Compare average bounding boxes
plot_avg_bbox_comparison(
    subject_id=0,
    model=model,
    root_dir=root_dir
)

**10. Real-Time Prediction Simulation and Metrics**

In [ ]:
def evaluate_real_time(model, subjects, root_dir):
    model.eval()
    device = next(model.parameters()).device
    errors = []
    times = []

    for subject_id in subjects:
        subject_str = subject_id
        img_dir = os.path.join(root_dir, 'openEDS', 'openEDS', subject_str)
        bbox_file = os.path.join(root_dir, 'bbox', 'bbox', f"{subject_str}.txt")

        with open(bbox_file, 'r') as f:
            bboxes = [list(map(float, line.strip().split())) for line in f.readlines()]

        transform = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])

        for idx in range(len(bboxes) - 1):
            img0 = Image.open(os.path.join(img_dir, f"{idx}.png")).convert('L')
            img1 = Image.open(os.path.join(img_dir, f"{idx+1}.png")).convert('L')

            img0_t = transform(img0)
            img1_t = transform(img1)
            diff = img1_t - img0_t
            input_tensor = torch.cat((img1_t, diff), dim=0).unsqueeze(0).to(device)

            gt_box = np.array(bboxes[idx + 1])

            start_time = time.time()
            with torch.no_grad():
                pred_box = model(input_tensor).squeeze(0).detach().cpu().numpy()
            end_time = time.time()

            elapsed_time = end_time - start_time
            times.append(elapsed_time)

            # Calculate Mean Squared Error for this frame
            mse = np.mean((pred_box - gt_box) ** 2)
            errors.append(mse)

    avg_error = np.mean(errors)
    avg_time = np.mean(times)

    print(f"Average Prediction Error (MSE): {avg_error}")
    print(f"Average Time Per Frame: {avg_time:.4f} seconds")

    return errors, times

In [ ]:
# Run real-time simulation and gather metrics
errors, times = evaluate_real_time(
    model=model,
    subjects=test_subjects,
    root_dir=root_dir
)